# Portfolio Exercise - Compare Recommender Strategies with MovieLens

In the lecture, you already implemented simple personalized recommender systems. In this portfolio exercise, the goal is **not to rebuild every algorithm from scratch**.

Instead, you will investigate a more realistic question:

> **How do different recommender strategies behave, and how should we evaluate them?**

You will compare three approaches:

1. **Popularity-based recommendation** - global interaction counts
2. **Content-based filtering** - similarity between a user profile and movie features
3. **User-based collaborative filtering** - similarity between users' rating behaviour

The content-based and collaborative-filtering implementations are provided. You will still write Python to inspect and transform data, build the baseline, compare rankings, evaluate the systems, and run a controlled experiment.

## What you should demonstrate

By the end of the notebook, you should be able to:

- identify users, items and interactions
- explain what evidence each recommender uses
- generate and compare Top-K rankings
- interpret **Precision@K** and **Recall@K**
- explain why chronological evaluation is preferable to a random split here
- reason about cold start, popularity bias and feedback loops

> **AI use is allowed.** You may use AI, documentation and your lecture material. During the **portfolio defence**, however, you must be able to explain your code, the recommendation results, the evaluation setup, and what you expect to happen when the data changes.


## What to submit

Submit **this completed notebook** in your portfolio. It should run from top to bottom after you complete the TODOs.

Keep written answers short and concrete. This is a technical portfolio artefact, not a report.


# 0. Dataset and setup

We use **MovieLens latest-small** from GroupLens. Ratings use a **0.5–5.0** scale and include timestamps. Movie information includes titles and genres.

The setup cell downloads the dataset the first time you run it.


In [1]:
from pathlib import Path
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.precision", 3)

DATA_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "ml-latest-small.zip"
EXTRACTED_DIR = DATA_DIR / "ml-latest-small"

DATA_DIR.mkdir(exist_ok=True)

if not EXTRACTED_DIR.exists():
    if not ZIP_PATH.exists():
        print("Downloading MovieLens latest-small...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

print("Dataset ready:", EXTRACTED_DIR)


Dataset ready: data/ml-latest-small


In [2]:
ratings = pd.read_csv(EXTRACTED_DIR / "ratings.csv")
movies = pd.read_csv(EXTRACTED_DIR / "movies.csv")
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")

print(f"Ratings: {len(ratings):,}")
print(f"Users:   {ratings['userId'].nunique():,}")
print(f"Movies:  {movies['movieId'].nunique():,}")
print(f"Period:  {ratings['datetime'].min().date()} to {ratings['datetime'].max().date()}")

display(ratings.head())
display(movies.head())


Ratings: 100,836
Users:   610
Movies:  9,742
Period:  1996-03-29 to 2018-09-24


,userId,movieId,rating,timestamp,datetime
0,1,1,4.0,964982703,2000-07-30 18:45:03
1,1,3,4.0,964981247,2000-07-30 18:20:47
2,1,6,4.0,964982224,2000-07-30 18:37:04
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


# 1. Create a chronological train/test split

A recommender should learn from **past behaviour** and be evaluated on **later behaviour**.

For every user, we therefore:

- sort interactions by timestamp
- use the earliest **80%** as training data
- use the latest **20%** as test data

This is a **per-user chronological split**. It is more realistic than randomly hiding ratings because later interactions for a user are not placed in that user's training history.

> For simplicity, this is **not one global time cut-off** across the complete dataset. Different users can therefore have different calendar-time split points.

Do not change the split code.

In [3]:
ratings_sorted = ratings.sort_values(
    ["userId", "timestamp", "movieId"]
).reset_index(drop=True)

ratings_sorted["interaction_nr"] = ratings_sorted.groupby("userId").cumcount()
ratings_sorted["n_user_interactions"] = ratings_sorted.groupby("userId")["movieId"].transform("size")
ratings_sorted["split_index"] = np.floor(
    0.80 * ratings_sorted["n_user_interactions"]
).astype(int)


train_mask = ratings_sorted["interaction_nr"] < ratings_sorted["split_index"]

ratings_train = (
    ratings_sorted.loc[train_mask, ratings.columns]
    .copy()
    .reset_index(drop=True)
)
ratings_test = (
    ratings_sorted.loc[~train_mask, ratings.columns]
    .copy()
    .reset_index(drop=True)
)

KNOWN_MOVIE_IDS = set(ratings_train["movieId"].unique())

print(f"Training interactions: {len(ratings_train):,}")
print(f"Test interactions:     {len(ratings_test):,}")
print(f"Movies known in train: {len(KNOWN_MOVIE_IDS):,}")


Training interactions: 80,419
Test interactions:     20,417
Movies known in train: 8,230


## Select users for the exercise

For evaluation, a test movie is considered **relevant** when the user rated it **4.0 or higher**.

To keep comparisons meaningful, we evaluate only test movies that already occur somewhere in the training data. This gives all three recommenders the same existing-item candidate universe; we discuss the **new-item cold-start** case separately later.

The code below selects users with enough history for the exercise. Choose one displayed `userId` as your `TARGET_USER`.


In [4]:
train_movie_counts = ratings_train.groupby("movieId").size()
popular_train_movie_ids = set(train_movie_counts[train_movie_counts >= 20].index)

train_counts = ratings_train.groupby("userId").size()
train_liked_popular = ratings_train[
    (ratings_train["rating"] >= 4.0)
    & ratings_train["movieId"].isin(popular_train_movie_ids)
]
train_liked_popular_counts = train_liked_popular.groupby("userId").size()
train_five_star_counts = ratings_train[
    ratings_train["rating"] == 5.0
].groupby("userId").size()

relevant_test_known = ratings_test[
    (ratings_test["rating"] >= 4.0)
    & ratings_test["movieId"].isin(KNOWN_MOVIE_IDS)
]
relevant_test_counts = relevant_test_known.groupby("userId").size()

eligible_users = pd.DataFrame({
    "train_interactions": train_counts,
    "liked_popular_train": train_liked_popular_counts,
    "relevant_test_items": relevant_test_counts,
    "five_star_train": train_five_star_counts,
}).fillna(0).astype(int)

eligible_users = eligible_users[
    (eligible_users["train_interactions"] >= 50)
    & (eligible_users["liked_popular_train"] >= 10)
    & (eligible_users["relevant_test_items"] >= 5)
    & (eligible_users["five_star_train"] >= 1)
]

if eligible_users.empty:
    raise RuntimeError(
        "No eligible users found. Check that MovieLens latest-small loaded correctly."
    )

eligible_sample = eligible_users.sample(
    min(15, len(eligible_users)), random_state=42
).sort_values("train_interactions", ascending=False)

display(eligible_sample)


,train_interactions,liked_popular_train,relevant_test_items,five_star_train
userId,,,,
307,780,76,87,11
66,276,144,43,53
573,239,168,57,59
325,216,67,30,26
20,193,65,26,33
198,184,90,14,49
552,150,49,8,6
479,144,37,16,26
326,121,45,23,17


In [5]:
# TODO Choose one userId from the table above.
# The default is valid, but you may select another displayed user.
TARGET_USER = int(eligible_sample.index[0])

print("Target user:", TARGET_USER)

Target user: 307


# 2. Understand your target user

Work only with the user's **training history**: these are the interactions the recommender is allowed to know.

Create `target_history` containing at least:

- `movieId`
- `title`
- `genres`
- `rating`
- `datetime`

Sort the user's highest ratings first.


In [6]:
# Create the target user's training history, sorted from highest to lowest rating.
target_history = (
    ratings_train[ratings_train["userId"] == TARGET_USER]
    .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
    [["movieId", "title", "genres", "rating", "datetime"]]
    .sort_values(by=["rating", "datetime"], ascending=[False, False])
    .reset_index(drop=True)
)

display(target_history.head(30))


,movieId,title,genres,rating,datetime
0,663,Kids in the Hall: Brain Candy (1996),Comedy,5.0,2007-09-10 20:53:20
1,8874,Shaun of the Dead (2004),Comedy|Horror,5.0,2007-09-10 20:45:56
2,4878,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,5.0,2007-08-15 15:35:47
3,7361,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,5.0,2007-08-03 17:28:37
4,919,"Wizard of Oz, The (1939)",Adventure|Children|Fantasy|Musical,5.0,2007-08-03 17:17:27
5,4239,Blow (2001),Crime|Drama,5.0,2007-08-02 20:32:26
6,1031,Bedknobs and Broomsticks (1971),Adventure|Children|Musical,5.0,2007-08-02 20:21:57
7,1215,Army of Darkness (1993),Action|Adventure|Comedy|Fantasy|Horror,5.0,2007-08-02 20:09:22
8,3535,American Psycho (2000),Crime|Horror|Mystery|Thriller,5.0,2007-08-02 20:01:55
9,2329,American History X (1998),Crime|Drama,5.0,2007-08-02 20:01:04


### Questions 1-2

**1. For MovieLens, identify:**

- **User:** A person who watches and rates movies, identified by `userId`.
- **Item:** A movie in the catalogue, identified by `movieId` (with a `title` and `genres`).
- **Interaction:** A user giving a star rating (from `0.5` to `5.0`) to a movie at a specific time (`timestamp`).
- **Explicit or implicit feedback?** **Explicit feedback**, because the user directly gives a star rating to show how much they liked or disliked the movie (instead of passive signals like clicks or watch time).

**2. Based on the training history above, describe one apparent preference of this user.** Refer to actual titles, genres or rating patterns.

**YOUR ANSWER (2-4 sentences):**  
User `307` strongly likes **Drama**, **Comedy**, and **Crime/Thriller** movies. In their training history, they gave **5.0 stars** to Drama and Crime movies like *Almost Famous (2000)*, *American History X (1998)*, *Blow (2001)*, and *Donnie Darko (2001)*, as well as comedies like *Shaun of the Dead (2004)* and *Kids in the Hall: Brain Candy (1996)*. They also gave **4.5 stars** to *The Departed (2006)*, *Sin City (2005)*, and *Walk the Line (2005)*.


# 3. Build a popularity baseline

Before personalisation, create a simple baseline yourself.

For this exercise:

> **popularity score = number of ratings a movie received in `ratings_train`**

The function must:

1. count ratings per movie
2. remove movies already rated by the target user
3. add movie information
4. rank by popularity score
5. return the Top-`n` unseen movies


In [7]:
def recommend_popularity(ratings_train, movies, target_user, n=20):
    """Recommend the most-rated unseen movies."""

    # 1. Count the number of ratings per movie in the training set
    movie_counts = (
        ratings_train.groupby("movieId")
        .size()
        .rename("score")
        .reset_index()
    )

    # 2. Find movieIds already rated by the target user
    seen_movie_ids = set(
        ratings_train.loc[ratings_train["userId"] == target_user, "movieId"]
    )

    # 3. Remove seen movies, add title and genres, sort by score, and keep Top-n
    unseen_movies = movie_counts[~movie_counts["movieId"].isin(seen_movie_ids)]

    ranking = (
        unseen_movies
        .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
        .sort_values(["score", "title"], ascending=[False, True])
        .head(n)
        .reset_index(drop=True)
    )

    return ranking


In [8]:
popularity_ranking = recommend_popularity(
    ratings_train, movies, TARGET_USER, n=20
)

display(popularity_ranking.head(5))


,movieId,score,title,genres
0,1198,176,Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981),Action|Adventure
1,2028,174,Saving Private Ryan (1998),Action|Drama|War
2,380,168,True Lies (1994),Action|Adventure|Comedy|Romance|Thriller
3,4226,137,Memento (2000),Mystery|Thriller
4,58559,128,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX


### Question 3 - baseline check

You removed movies the target user has already rated. Why is this recommender nevertheless **not personalised**?

**YOUR ANSWER (1-2 sentences):**  
The score of every movie comes only from the total number of ratings across **all users**, not from this user's own taste. Removing seen movies only hides movies the user already watched; the order of all remaining movies is still the exact same for everyone.


# 4. Generate personalised recommendations

You already implemented personalised recommenders during the lecture exercises, so the implementations are supplied here.

We use:

- **Content-based filtering:** create a genre profile from movies the target user rated **4.0 or higher**, then rank unseen movies by **cosine similarity** to that profile.
- **User-based collaborative filtering:** find users with small **average absolute rating differences** on movies both users rated, then use those neighbours' ratings to rank unseen movies.

The collaborative method deliberately does **not** use raw cosine similarity on 1-5 ratings (the lecture showed why cosine can be misleading for this type of explicit rating data).


In [9]:
def recommend_content_based(
    ratings_train,
    movies,
    target_user,
    n=20,
    genre_matrix=None,
    candidate_movie_ids=None,
):
    """Genre-based user profile + cosine similarity."""

    if genre_matrix is None:
        genre_matrix = movies.set_index("movieId")["genres"].str.get_dummies("|")

    target_ratings = ratings_train[ratings_train["userId"] == target_user]
    liked_ids = target_ratings.loc[target_ratings["rating"] >= 4.0, "movieId"]
    liked_ids = [movie_id for movie_id in liked_ids if movie_id in genre_matrix.index]

    if len(liked_ids) == 0:
        raise ValueError("Target user has no liked movies in the training data.")

    profile = genre_matrix.loc[liked_ids].mean(axis=0)
    user_profile = profile.to_numpy().reshape(1, -1)

    seen_ids = set(target_ratings["movieId"])

    if candidate_movie_ids is None:
        candidate_movie_ids = set(ratings_train["movieId"].unique())

    candidate_ids = [
        movie_id
        for movie_id in candidate_movie_ids
        if movie_id not in seen_ids and movie_id in genre_matrix.index
    ]

    candidate_features = genre_matrix.loc[candidate_ids]
    scores = cosine_similarity(candidate_features, user_profile).ravel()

    ranking = pd.DataFrame({
        "movieId": candidate_features.index,
        "score": scores,
    })

    ranking = (
        ranking
        .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
        .sort_values(["score", "title"], ascending=[False, True])
        .head(n)
        .reset_index(drop=True)
    )

    return ranking, profile.sort_values(ascending=False)


In [10]:
def recommend_user_based(
    ratings_train,
    movies,
    target_user,
    n=20,
    min_common=5,
    n_neighbours=10,
    min_support=2,
):
    """User-based CF using average absolute rating difference on common movies."""

    target = (
        ratings_train[ratings_train["userId"] == target_user]
        [["movieId", "rating"]]
        .rename(columns={"rating": "target_rating"})
    )

    if target.empty:
        raise ValueError("Target user is not present in the training data.")

    overlap = (
        ratings_train[
            (ratings_train["userId"] != target_user)
            & ratings_train["movieId"].isin(target["movieId"])
        ]
        .merge(target, on="movieId", how="inner")
    )


    overlap["abs_diff"] = (
        overlap["rating"] - overlap["target_rating"]
    ).abs()

    neighbours = (
        overlap.groupby("userId")
        .agg(
            distance=("abs_diff", "mean"),
            common_ratings=("movieId", "size"),
        )
        .reset_index()
    )

    neighbours = (
        neighbours[neighbours["common_ratings"] >= min_common]
        .sort_values(["distance", "common_ratings"], ascending=[True, False])
        .head(n_neighbours)
        .reset_index(drop=True)
    )

    if neighbours.empty:
        raise ValueError("No suitable neighbours found for this user.")

    neighbour_ids = neighbours["userId"].tolist()
    seen_ids = set(target["movieId"])

    candidate_ratings = ratings_train[
        ratings_train["userId"].isin(neighbour_ids)
        & ~ratings_train["movieId"].isin(seen_ids)
    ]

    predictions = (
        candidate_ratings.groupby("movieId")
        .agg(score=("rating", "mean"), support=("rating", "size"))
        .reset_index()
    )

    predictions = predictions[predictions["support"] >= min_support]

    ranking = (
        predictions
        .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
        .sort_values(
            ["score", "support", "title"],
            ascending=[False, False, True],
        )
        .head(n)
        .reset_index(drop=True)
    )

    return ranking, neighbours


In [11]:
GENRE_MATRIX = movies.set_index("movieId")["genres"].str.get_dummies("|")

content_ranking, content_profile = recommend_content_based(
    ratings_train,
    movies,
    TARGET_USER,
    n=20,
    genre_matrix=GENRE_MATRIX,
    candidate_movie_ids=KNOWN_MOVIE_IDS,
)

usercf_ranking, neighbours = recommend_user_based(
    ratings_train, movies, TARGET_USER, n=20
)

print("Content-based Top-5")
display(content_ranking.head(5))

print("Strongest genre signals in the target user's content profile")
display(content_profile.head(8).rename("profile_weight").to_frame())

print("User-based CF Top-5")
display(usercf_ranking.head(5))

print("Nearest users used by collaborative filtering")
display(neighbours.head(10))


Content-based Top-5
Strongest genre signals in the target user's content profile
User-based CF Top-5
Nearest users used by collaborative filtering


,movieId,score,title,genres
0,3893,0.835,Nurse Betty (2000),Comedy|Crime|Drama|Romance|Thriller
1,1912,0.835,Out of Sight (1998),Comedy|Crime|Drama|Romance|Thriller
2,970,0.830,Beat the Devil (1953),Adventure|Comedy|Crime|Drama|Romance
3,6023,0.812,Band of Outsiders (Bande à part) (1964),Comedy|Crime|Drama|Romance
4,72919,0.812,Did You Hear About the Morgans? (2009),Comedy|Crime|Drama|Romance


,profile_weight
Comedy,0.490
Drama,0.438
Crime,0.240
Romance,0.219
Thriller,0.208
Adventure,0.198
Fantasy,0.198
Children,0.125


,movieId,score,support,title,genres
0,109487,4.750,2,Interstellar (2014),Sci-Fi|IMAX
1,116797,4.500,3,The Imitation Game (2014),Drama|Thriller|War
2,115713,4.500,2,Ex Machina (2015),Drama|Sci-Fi|Thriller
3,59315,4.500,2,Iron Man (2008),Action|Adventure|Sci-Fi
4,1203,4.333,3,12 Angry Men (1957),Drama


,userId,distance,common_ratings
0,550,0.300,5
1,338,0.393,14
2,473,0.450,10
3,65,0.455,11
4,490,0.480,25
5,401,0.500,21
6,148,0.500,9
7,152,0.543,23
8,123,0.567,15
9,23,0.568,22


## Compare the three Top-5 rankings

Build one table with the movie titles at ranks 1-5 for each approach (popularity, content-based and user-based CF).

Do **not** type movie names manually; construct the table from the ranking DataFrames.


In [12]:
# Build a 5-row comparison table from the three ranking DataFrames.
comparison = pd.DataFrame({
    "Rank": range(1, 6),
    "Popularity": popularity_ranking["title"].head(5).values,
    "Content-based": content_ranking["title"].head(5).values,
    "User-based CF": usercf_ranking["title"].head(5).values,
}).set_index("Rank")

display(comparison)


,Popularity,Content-based,User-based CF
Rank,,,
1,Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981),Nurse Betty (2000),Interstellar (2014)
2,Saving Private Ryan (1998),Out of Sight (1998),The Imitation Game (2014)
3,True Lies (1994),Beat the Devil (1953),Ex Machina (2015)
4,Memento (2000),Band of Outsiders (Bande à part) (1964),Iron Man (2008)
5,"Dark Knight, The (2008)",Did You Hear About the Morgans? (2009),12 Angry Men (1957)


In [13]:
# Evidence that can help you explain the highest-ranked CF recommendation.
top_cf_movie_id = int(usercf_ranking.iloc[0]["movieId"])

top_cf_neighbour_evidence = (
    ratings_train[
        ratings_train["userId"].isin(neighbours["userId"])
        & (ratings_train["movieId"] == top_cf_movie_id)
    ]
    .merge(neighbours, on="userId", how="left")
    [["userId", "rating", "distance", "common_ratings"]]
    .sort_values("distance")
)

print("Neighbour ratings for the highest-ranked user-CF movie:")
display(top_cf_neighbour_evidence)


Neighbour ratings for the highest-ranked user-CF movie:


,userId,rating,distance,common_ratings
1,550,5.0,0.300,5
0,123,4.5,0.567,15


### Question 4 - compare the evidence

**a.** Does any movie appear in more than one Top-5? Give one example if so.  
No movie appears in more than one Top-5 list for User `307`. Each method picks completely different movies because each method uses different evidence.

**b.** Choose one highly ranked **content-based** recommendation. Why did it receive a high score? Refer to the user's genre profile and the movie's features.  
*Nurse Betty (2000)* (and *Out of Sight (1998)*) is ranked #1 with a cosine similarity score of `0.835`. User `307`'s top liked genres in their profile are **Comedy (`0.490`)**, **Drama (`0.438`)**, **Crime (`0.240`)**, **Romance (`0.219`)**, and **Thriller (`0.208`)**, and *Nurse Betty* has all five of those exact genres (`Comedy|Crime|Drama|Romance|Thriller`).

**c.** Explain why the **highest-ranked user-based CF movie** could receive a high score. Refer to the neighbours and their rating behaviour.  
*Interstellar (2014)* is ranked #1 with a predicted score of `4.75`. It got this high score because two of User `307`'s closest neighbours—users who gave very similar ratings on shared movies (`userId=550` with average difference `0.300` and `userId=123` with average difference `0.567`)—rated *Interstellar* `5.0` and `4.5`, meeting the minimum support of 2 neighbours.

**d.** Why can content-based filtering and collaborative filtering produce different rankings for the same user?  
Content-based filtering compares **movie features (genres)** with the user's liked genres, without looking at other users. Collaborative filtering ignores genres completely and instead looks at **what similar users rated highly**, so it can recommend movies with fewer genre tags (like `Sci-Fi|IMAX` for *Interstellar*) if neighbours loved them.


# 5. Evaluate the three recommenders for your target user

We now evaluate on the user's **later interactions** from `ratings_test`.

A test movie is relevant when:

- the target user rated it **4.0 or higher**; and
- the movie already existed in the training candidate universe.

We use **Top-10** recommendations and calculate:

- **Hits@10**: number of relevant test movies retrieved
- **Precision@10**: relevant items in the Top-10 / 10
- **Recall@10**: relevant items retrieved / all relevant test items


In [14]:
def precision_at_k(recommended_ids, relevant_ids, k):
    top_k = list(recommended_ids)[:k]
    hits = sum(movie_id in relevant_ids for movie_id in top_k)
    return hits / k


def recall_at_k(recommended_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0
    top_k = list(recommended_ids)[:k]
    hits = sum(movie_id in relevant_ids for movie_id in top_k)
    return hits / len(relevant_ids)


def relevant_test_items_for_user(user_id):
    rows = ratings_test[
        (ratings_test["userId"] == user_id)
        & (ratings_test["rating"] >= 4.0)
        & (ratings_test["movieId"].isin(KNOWN_MOVIE_IDS))
    ]
    return set(rows["movieId"])


K_EVAL = 10
RELEVANT_TEST_ITEMS = relevant_test_items_for_user(TARGET_USER)
print("Relevant later test items for target user:", len(RELEVANT_TEST_ITEMS))


Relevant later test items for target user: 87


In [15]:
rankings = {
    "Popularity": popularity_ranking,
    "Content-based": content_ranking,
    "User-based CF": usercf_ranking,
}

# Create one row per method with Hits@10, Precision@10 and Recall@10.
rows = []

for method, ranking in rankings.items():
    recommended_ids = ranking["movieId"].head(K_EVAL).tolist()
    hits = sum(movie_id in RELEVANT_TEST_ITEMS for movie_id in recommended_ids)
    precision = precision_at_k(recommended_ids, RELEVANT_TEST_ITEMS, K_EVAL)
    recall = recall_at_k(recommended_ids, RELEVANT_TEST_ITEMS, K_EVAL)

    rows.append({
        "method": method,
        f"Hits@{K_EVAL}": hits,
        f"Precision@{K_EVAL}": precision,
        f"Recall@{K_EVAL}": recall,
    })

target_evaluation = pd.DataFrame(rows)
display(target_evaluation)


,method,Hits@10,Precision@10,Recall@10
0,Popularity,4,0.4,0.046
1,Content-based,0,0.0,0.000
2,User-based CF,2,0.2,0.023


In [16]:
# Reveal the relevant later interactions after you have calculated the metrics.
relevant_test_movies = (
    ratings_test[
        (ratings_test["userId"] == TARGET_USER)
        & (ratings_test["movieId"].isin(RELEVANT_TEST_ITEMS))
    ]
    .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
    [["movieId", "title", "genres", "rating", "datetime"]]
    .sort_values("datetime")
)

display(relevant_test_movies)


,movieId,title,genres,rating,datetime
0,48082,"Science of Sleep, The (La science des rêves) (2006)",Comedy|Drama|Fantasy|Romance,4.5,2007-09-11 16:44:06
1,41571,Memoirs of a Geisha (2005),Drama|Romance,4.5,2007-09-12 14:15:46
2,51935,Shooter (2007),Action|Drama|Thriller,4.0,2007-09-12 14:44:39
3,6724,Paper Moon (1973),Comedy|Crime|Drama,4.0,2007-09-12 14:50:35
4,1247,"Graduate, The (1967)",Comedy|Drama|Romance,4.5,2007-09-13 13:10:59
...,...,...,...,...,...
82,3147,"Green Mile, The (1999)",Crime|Drama,4.0,2009-05-20 13:24:00
83,2000,Lethal Weapon (1987),Action|Comedy|Crime|Drama,4.0,2009-05-20 13:24:14
84,68954,Up (2009),Adventure|Animation|Children|Drama,5.0,2009-06-01 19:32:58
85,69122,"Hangover, The (2009)",Comedy|Crime,5.0,2009-06-08 15:56:56


### Question 5 - interpret one-user evaluation

**a.** Which method retrieved the most relevant later movies for this user?  
**Popularity** retrieved the most relevant test movies (`Hits@10 = 4`, `Precision@10 = 0.400`, `Recall@10 = 0.046`), followed by **User-based CF** (`Hits@10 = 2`, `Precision@10 = 0.200`), while **Content-based** had `0` hits in the Top-10.

**b.** Does that prove this method is generally the best recommender? Why (not)?  
No. This is only **one user**, which is too small a sample to judge a whole system. Also, MovieLens users often watch famous blockbusters next, which helps Popularity on offline historical data; we need to average metrics (like MAP@K) across many users and test online.

**c.** Why is this chronological evaluation more realistic than randomly hiding ratings from the user's history?  
In real life, a recommender only knows **past ratings** and must predict **future ratings**. A random split leaks future ratings into the training set and asks the model to "predict" the past, which is unrealistic.

**d.** If a recommended movie is not in `RELEVANT_TEST_ITEMS`, does that necessarily mean the user dislikes it? Explain.  
No. `RELEVANT_TEST_ITEMS` only contains movies the user **actually watched and rated** during the test period. If a recommended movie is missing from that set, the user probably just never saw or watched it yet—they might still love it if they watched it.


# 6. Predict the effect of changing one rating

Now test whether you understand what data each method uses.

Choose one movie that the target user rated **5 stars in the training data** (for example, *Eternal Sunshine of the Spotless Mind (2004)*, `movieId = 7361`). Imagine that the user had rated that same movie **1 star instead**.

## Predict without running anything

Complete this table:

| Method | Could its ranking change? | Why? |
|---|---|---|
| **Popularity** | **No** | Popularity only counts **how many** ratings each movie got in `ratings_train`, not the star value (`5` vs `1`). Since the user still rated the movie, it is still removed as "already seen", and all other movie counts stay the same. |
| **Content-based** | **Yes** | The user profile is the average of genres from movies the user rated **4.0 or higher**. Changing a rating from `5` to `1` drops that movie out of the liked set (`>= 4.0`), which changes the user's genre profile and cosine similarity scores. |
| **User-based CF** | **Yes** | Neighbour similarity is based on the **average absolute difference** in ratings (`|r_u - r_v|`) on shared movies. Changing a `5` to a `1` changes the rating difference with other users, which can change the top 10 nearest neighbours and their predicted scores. |


In [17]:
# Controlled experiment: change one 5-star rating (Eternal Sunshine of the Spotless Mind, movieId=7361) to 1.0
CHANGED_MOVIE_ID = 7361
ratings_train_modified = ratings_train.copy()
mask = (ratings_train_modified["userId"] == TARGET_USER) & (ratings_train_modified["movieId"] == CHANGED_MOVIE_ID)
ratings_train_modified.loc[mask, "rating"] = 1.0

pop_mod = recommend_popularity(ratings_train_modified, movies, TARGET_USER, n=20)
cb_mod, _ = recommend_content_based(
    ratings_train_modified, movies, TARGET_USER, n=20, genre_matrix=GENRE_MATRIX, candidate_movie_ids=KNOWN_MOVIE_IDS
)
cf_mod, _ = recommend_user_based(ratings_train_modified, movies, TARGET_USER, n=20)

print("Did Top-20 Popularity change?    ", not popularity_ranking["movieId"].equals(pop_mod["movieId"]))
print("Did Top-20 Content-based change? ", not content_ranking["movieId"].equals(cb_mod["movieId"]))
print("Did Top-20 User-based CF change? ", not usercf_ranking["movieId"].equals(cf_mod["movieId"]))


Did Top-20 Popularity change?     False
Did Top-20 Content-based change?  True
Did Top-20 User-based CF change?  True


# 7. Difficult situations

## Scenario A - new movie

A brand-new movie enters the catalogue. You know its **title and genres**, but **nobody has rated it yet**.

| Approach | Can meaningfully recommend the new movie immediately? | Why? |
|---|---|---|
| **Popularity** | **No** | Nobody has rated it yet, so its rating count is `0`. It will sit at the very bottom of the ranking. |
| **Content-based** | **Yes** | It only needs the movie's **genres** (features) to compute cosine similarity with the user's genre profile. It does not need any existing ratings (**no item cold-start**). |
| **User-based CF** | **No** | Collaborative filtering needs neighbours to have rated the movie first (at least `min_support = 2` neighbour ratings). With `0` ratings, it cannot compute a score (**item cold-start**). |

> Note: for the earlier offline comparison, we deliberately restricted all methods to movies already known in training. Here we are explicitly asking about a **new item**.

## Scenario B - feedback loop and popularity bias

Suppose the platform repeatedly gives its highest-ranked movies much more exposure than other movies.

**How could this create a feedback loop and reinforce popularity bias?**

**YOUR ANSWER (2-3 sentences):**  
Users mostly click, watch, and rate the movies shown at the top of their screen. Because popular movies get shown the most, they collect even more ratings, while good niche movies stay hidden and get almost no ratings. When the recommender retrains on this new data, the popular movies get an even higher score, creating a "rich-get-richer" feedback loop (**popularity bias**).


# 8. Final conclusion

Complete each statement in **one sentence**.

**Popularity-based recommendation is useful when...**  
you have a **brand-new user** with no rating history (**user cold-start**) or when you need a fast, simple baseline to compare smarter models against.

**Content-based recommendation is useful when...**  
you have clear **item features** (like genres or tags) and want to recommend **brand-new movies** right away without waiting for other users to rate them.

**Collaborative filtering is useful when...**  
you have plenty of user ratings and want to help users discover great movies outside their usual genres based on **what similar users enjoyed**.

Finally:

> What is **one thing you would investigate next** before deciding which recommender to deploy?

**YOUR ANSWER (1–2 sentences):**  
I would evaluate all three methods (and a **hybrid model** combining them) across **all users** in the test set using **MAP@10** instead of looking at just one user. After that, I would run an **online A/B test** to see which recommender gets higher real-world engagement and user satisfaction.


# Portfolio defence - be ready to explain

You do **not** need to write answers to these questions in advance, but you should be able to answer questions such as:

- Why is a recommender mainly a **ranking** problem here?
- Who are the users, items and interactions in MovieLens?
- Is the feedback implicit or explicit? Why?
- Why is the popularity baseline still not personalised after removing seen movies?
- What exactly does content-based filtering compare?
- What exactly does user-based collaborative filtering compare?
- Why did a particular movie appear in one Top-5 but not another?
- What do Precision@10 and Recall@10 mean in this notebook?
- Why do we train on earlier interactions and test on later ones?
- Why should you not conclude that one method is generally best from one user?
- What happens for a new movie with features but no ratings?
- How can recommendation exposure create a feedback loop and reinforce popularity bias?

## Submission checklist

Before submitting, check that:

- [ ] the notebook runs from top to bottom after you complete the TODOs
- [ ] `TARGET_USER` is set to an eligible user
- [ ] `target_history` is created with Python
- [ ] your popularity function returns only the Top-`n` unseen movies ranked by rating count
- [ ] all three Top-5 rankings are displayed side by side
- [ ] your target-user evaluation contains Hits@10, Precision@10 and Recall@10
- [ ] the cold-start and feedback-loop/popularity-bias questions are completed
- [ ] all required short-answer cells are completed
- [ ] you can explain the code and results during the portfolio defence